# Analiza Forex - Midas Project

Notebook do analizy danych przy użyciu modułów Midas (Analysis, Database, DataConnector).

In [1]:
import plotly.graph_objects as go
import pandas as pd
import config
from data_connector import MT5Connector
from database import DatabaseManager
from analysis import Analyzer

# Ustawienia wyświetlania
pd.set_option('display.max_columns', None)

## 1. Połączenie i Pobranie Danych

In [2]:
connector = MT5Connector()
db = DatabaseManager()

symbol = 'XAUUSD'

df = db.load_candles(symbol, limit=10000)

if df.empty:
    print("Brak danych. Upewnij się, że uruchomiłeś main.py.")
else:
    print(f"Załadowano {len(df)} świec dla {symbol}.")
    print(df.tail())

## 2. Analiza (ZigZag + Swingi)

In [3]:
if not df.empty:
    df['zigzag'] = Analyzer.calculate_zigzag(df)
    swings = Analyzer.analyze_swings(df)
    
    print("Znalezione swingi:")
    print(swings[['start_time', 'end_time', 'direction', 'range', 'volume', 'duration_mins', 'intensity']].tail())

## 3. Predykcja & Anomalie (SOT, Hinge)

In [4]:
# 3.1 NN Prediction
prediction = None
if not swings.empty:
    prediction = Analyzer.predict_next_swing_nn(swings, k=5)
    if prediction:
        print("\n=== PROGNOZA (Nearest Neighbors) ===")
        print(f"Kierunek: {prediction['direction']}")
        print(f"Cel cenowy: {prediction['target_price']:.2f}")

# 3.2 Wave Logic Signals
sot_signals = pd.DataFrame()
hinge_signals = pd.DataFrame()

if not swings.empty:
    # Shortening of the Thrust
    sot_signals = Analyzer.detect_sot(swings)
    if not sot_signals.empty:
        print(f"\n=== WYKRYTO SOT ({len(sot_signals)} sygnałów) ===")
        display(sot_signals.tail(3))
        
    # Hinge / Apex (Dullness)
    hinge_signals = Analyzer.detect_hinge(swings)
    if not hinge_signals.empty:
        print(f"\n=== WYKRYTO HINGE/APEX ({len(hinge_signals)} sygnałów) ===")
        display(hinge_signals.tail(3))

## 4. Backtesting Compartment

In [ ]:
# Backtest Configuration
BACKTEST_MODE = True
BACKTEST_METHOD = 'NN'
K_NEIGHBORS = 5

backtest_results = pd.DataFrame()

if BACKTEST_MODE:
    if BACKTEST_METHOD == 'NN':
        print(f"Running NN Backtest (k={K_NEIGHBORS})...")
        backtest_results = Analyzer.backtest_nn(swings, k=K_NEIGHBORS)
        
        if not backtest_results.empty:
            avg_price_err = backtest_results['price_error'].mean()
            avg_range_err = backtest_results['range_error'].mean()
            avg_dur_err = backtest_results['duration_error'].mean()
            
            print(f"Backtest Complete. Avg PrErr: {avg_price_err:.2f}, Avg RgErr: {avg_range_err:.2f}, Avg DurErr: {avg_dur_err:.1f}m")
            display(backtest_results.tail(3))
    else:
        print(f"Method {BACKTEST_METHOD} not implemented.")

## 5. View & Visualization

In [ ]:
if not df.empty:
    fig = go.Figure()

    # View limit
    display_start = max(0, len(df) - 600)
    df_v = df.iloc[display_start:]
    
    # 1. Candlesticks
    fig.add_trace(go.Candlestick(
        x=df_v['time'], open=df_v['open'], high=df_v['high'], low=df_v['low'], close=df_v['close'],
        name='Price', opacity=0.4
    ))

    # 2. Actual ZigZag (Blue)
    zz_pts = df_v[df_v['zigzag'] != 0]
    fig.add_trace(go.Scatter(
        x=zz_pts['time'], y=zz_pts['zigzag'], mode='lines+markers', 
        line=dict(color='blue', width=2), name='ZigZag'
    ))

    # 3. Live Prediction (Magenta)
    if prediction:
        fig.add_trace(go.Scatter(
            x=[swings.iloc[-1]['end_time'], prediction['target_time']],
            y=[swings.iloc[-1]['end_price'], prediction['target_price']],
            mode='lines+markers', line=dict(color='magenta', width=3, dash='dot'), name='Live Prediction'
        ))

    # 4. Backtest Predictions (Yellow)
    if not backtest_results.empty:
        visible_bt = backtest_results[backtest_results['start_time'] >= df_v['time'].iloc[0]]
        bt_x, bt_y = [], []
        for _, r in visible_bt.iterrows():
            bt_x.extend([r['start_time'], r['target_time'], None])
            bt_y.extend([r['start_price'], r['target_price'], None])
        fig.add_trace(go.Scatter(
            x=bt_x, y=bt_y, mode='lines', line=dict(color='yellow', width=1, dash='dot'), 
            name='Hist Pred (Yellow)', opacity=0.4
        ))

    # 5. SOT Signals (Triangles)
    if not sot_signals.empty:
        visible_sot = sot_signals[sot_signals['time'] >= df_v['time'].iloc[0]]
        for _, s in visible_sot.iterrows():
            color = 'green' if s['direction'] == 'Up' else 'red'
            symbol_marker = 'triangle-up' if s['direction'] == 'Up' else 'triangle-down'
            fig.add_trace(go.Scatter(
                x=[s['time']], y=[s['price']], mode='markers',
                marker=dict(color=color, size=14, symbol=symbol_marker, line=dict(color='white', width=1)),
                name=f"SOT {s['direction']}"
            ))
            
    # 6. Hinge / Apex (Diamonds)
    if not hinge_signals.empty:
        visible_hinge = hinge_signals[hinge_signals['time'] >= df_v['time'].iloc[0]]
        if not visible_hinge.empty:
            fig.add_trace(go.Scatter(
                x=visible_hinge['time'], y=visible_hinge['price'], mode='markers',
                marker=dict(color='white', size=16, symbol='diamond', line=dict(color='cyan', width=2)),
                name="Hinge / Apex (Dullness)"
            ))

    fig.update_layout(
        title=f'{symbol} Midas Analysis (Advanced Wave Logic)',
        template='plotly_dark', xaxis_rangeslider_visible=False, height=800
    )
    fig.show()